# 01 — Embedding Exploration: Gemini API vs FastEmbed

This notebook explores the same real job-description texts with two embedding approaches:

- **Gemini API** — the embedding provider used by the deployed SmartHire application.
- **FastEmbed (`BAAI/bge-small-en-v1.5`)** — a local neural embedding model used here only for comparison on the developer laptop.

The notebook uses a small sample so the comparison is inexpensive and easy to inspect.

In [6]:
import sys
print(sys.executable)

c:\Users\harsh\Downloads\Final Gen ai project output\smarthire-genai-final\.venv\Scripts\python.exe


In [7]:
import fastembed
print("FastEmbed:", fastembed.__version__)

c:\Users\harsh\Downloads\Final Gen ai project output\smarthire-genai-final\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FastEmbed: 0.8.0


In [1]:
from pathlib import Path
import sys

# Find the SmartHire project root so these notebooks work whether Jupyter
# is launched from the project root or from the notebooks/ directory.
HERE = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [HERE, *HERE.parents]:
    if (candidate / "src" / "config.py").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the SmartHire project root. "
        "Open this notebook from inside the project repository."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\harsh\Downloads\Final Gen ai project output\smarthire-genai-final


In [2]:
import numpy as np
import pandas as pd

from src.core.paths import JOBS_CSV

jobs = pd.read_csv(JOBS_CSV)

print(f"Dataset: {JOBS_CSV}")
print(f"Rows available: {len(jobs):,}")
print("Columns:", list(jobs.columns))

Dataset: C:\Users\harsh\Downloads\Final Gen ai project output\smarthire-genai-final\data\jobs\smarthire_jobs_80.csv
Rows available: 80
Columns: ['job_id', 'category', 'title', 'company', 'location', 'experience', 'salary', 'skills', 'description', 'jdURL']


## Select five real job descriptions

Two Software Engineer-family postings are selected as the clearly similar pair. One Software Engineer posting and one Data Scientist posting form the clearly different pair. A fifth job provides an additional comparison example.

In [3]:
def pick_first(frame, mask, label):
    selected = frame.loc[mask].iloc[0]
    return selected

software_mask = jobs["title"].astype(str).str.contains(
    r"software engineer|software development engineer",
    case=False,
    regex=True,
    na=False,
)
data_scientist_mask = jobs["title"].astype(str).str.contains(
    r"data scientist",
    case=False,
    regex=True,
    na=False,
)

similar_a = pick_first(jobs, software_mask, "software A")
remaining_software = jobs.loc[
    software_mask & (jobs.index != similar_a.name)
]
similar_b = remaining_software.iloc[0]

different_a = similar_a
different_b = pick_first(jobs, data_scientist_mask, "data scientist")

# Fifth text: another role with a technical but distinct profile.
extra_mask = jobs["title"].astype(str).str.contains(
    r"data analyst|business analyst|machine learning engineer|data engineer",
    case=False,
    regex=True,
    na=False,
)
extra = pick_first(jobs, extra_mask, "additional role")

sample_rows = [similar_a, similar_b, different_b, extra, jobs.iloc[0]]
sample = pd.DataFrame(sample_rows).drop_duplicates(subset=["job_id"])

texts = sample["description"].fillna("").astype(str).tolist()

for i, row in sample.iterrows():
    print("=" * 80)
    print(f"{row['title']} — {row['company']}")
    print(f"Category: {row['category']}")
    print(row["description"][:1200].strip())
    print()

Software Engineer — Hsbc
Category: Software Engineer
Experience in working on integration between cloud systems and on-perm application . Requirements . To be successful in this role,you should meet the following requirements: . Applicants should have minimum 2 end to end enterprise web application design and development project experience . Hands on experience working with middleware,APIs . .

Software Engineer — Pike Automation
Category: Software Engineer
The ideal candidate will have extensive experience with software technologies,including C#,C++,.NET,SQL Server,Oracle PL / SQL etc. . Key Responsibilities: Automation System Design and Implementation: - Design,develop,and implement automation systems using .NET technologies Bachelors (Required) total work: 3 years (Preferred) coding: 3 years (Preferred)

Data Scientist — Capgemini
Category: Data Scientist
. Good problem solver with a skill of solving business problem with analytics and data as primary means Excellent coding experien

## Embed with Gemini API

The production application uses Gemini API embeddings. Gemini Embedding 2 can return separate embeddings when each input is wrapped in its own `Content` object. The helper below mirrors that behavior and sends the small sample as one batch.

In [4]:
from google import genai
from google.genai import types
from src import config

if not config.GEMINI_API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY is not configured. "
        "Create .env from .env.example and add your key."
    )

gemini_client = genai.Client(api_key=config.GEMINI_API_KEY)

gemini_contents = [
    types.Content(
        parts=[types.Part.from_text(text=text)]
    )
    for text in texts
]

gemini_result = gemini_client.models.embed_content(
    model=config.GEMINI_EMBEDDING_MODEL,
    contents=gemini_contents,
    config=types.EmbedContentConfig(
        output_dimensionality=config.GEMINI_EMBEDDING_DIMENSION,
    ),
)

gemini_vectors = np.asarray(
    [item.values for item in gemini_result.embeddings],
    dtype="float32",
)

# Normalize before cosine comparisons.
gemini_norms = np.linalg.norm(gemini_vectors, axis=1, keepdims=True)
gemini_vectors = gemini_vectors / np.clip(gemini_norms, 1e-12, None)

print("Gemini model:", config.GEMINI_EMBEDDING_MODEL)
print("Gemini vector shape:", gemini_vectors.shape)

Gemini model: gemini-embedding-2
Gemini vector shape: (4, 768)


## Embed with local FastEmbed

This backend is **local, heavier — intended for local notebook comparisons, use with caution if deployed**.

It is not part of the deployed application's default path.

In [8]:
import importlib

try:
    fastembed_module = importlib.import_module("fastembed")
    TextEmbedding = fastembed_module.TextEmbedding
except Exception as exc:
    raise RuntimeError(
        "FastEmbed is not installed in this virtual environment. "
        "Install it locally with: python -m pip install fastembed"
    ) from exc

fastembed_model = TextEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    threads=1,
)

fastembed_vectors = np.asarray(
    list(
        fastembed_model.embed(
            texts,
            batch_size=64,
        )
    ),
    dtype="float32",
)

fastembed_norms = np.linalg.norm(
    fastembed_vectors,
    axis=1,
    keepdims=True,
)
fastembed_vectors = fastembed_vectors / np.clip(
    fastembed_norms,
    1e-12,
    None,
)

print("FastEmbed model: BAAI/bge-small-en-v1.5")
print("FastEmbed vector shape:", fastembed_vectors.shape)

FastEmbed model: BAAI/bge-small-en-v1.5
FastEmbed vector shape: (4, 384)


## Compare cosine similarity

Higher cosine similarity means the two vectors are closer in their respective embedding spaces.

In [ ]:
def cosine_score(vectors, i, j):
    return float(np.dot(vectors[i], vectors[j]))

# Pair A: clearly similar — two Software Engineer-family postings.
similar_score_gemini = cosine_score(gemini_vectors, 0, 1)
similar_score_fastembed = cosine_score(fastembed_vectors, 0, 1)

# Pair B: clearly different — Software Engineer vs Data Scientist.
different_score_gemini = cosine_score(gemini_vectors, 0, 2)
different_score_fastembed = cosine_score(fastembed_vectors, 0, 2)

print("SIMILAR PAIR")
print(f"  Job 1: {sample.iloc[0]['title']}")
print(f"  Job 2: {sample.iloc[1]['title']}")
print(f"  Gemini cosine similarity:    {similar_score_gemini:.4f}")
print(f"  FastEmbed cosine similarity: {similar_score_fastembed:.4f}")
print()

print("DIFFERENT PAIR")
print(f"  Job 1: {sample.iloc[0]['title']}")
print(f"  Job 2: {sample.iloc[2]['title']}")
print(f"  Gemini cosine similarity:    {different_score_gemini:.4f}")
print(f"  FastEmbed cosine similarity: {different_score_fastembed:.4f}")

## Observations

Interpret the numbers as a **relative comparison**, not as an absolute quality score.

Look for the pattern you would expect from a useful semantic embedding model:

- The clearly similar Software Engineer pair should receive a higher similarity score.
- The Software Engineer vs Data Scientist pair should receive a lower score.
- The two models may produce different numerical scores because they use different embedding spaces and model training.

**Deployment note:** SmartHire's deployed application uses the **Gemini API embedding provider** (`EMBEDDING_PROVIDER=gemini-api`). FastEmbed is used in this notebook only to compare a local neural model on the developer laptop; it is not the deployment default.